# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My rule and its reason codes

*Write the rule in plain words first. Then the reason codes it can output.*

---

**My rule, in plain words:** a page deserves refresh review first when people can still
see it, nobody has updated it for a long time, and where it sits in the results leaves
something worth recovering. The more visible and the longer untouched, the earlier it
goes in the queue. Everything in the rule is observable at review time — impressions,
position, CTR, word count, content age, days since last update. Nothing in it is fitted,
and nothing in it is derived from the label (see Section 4 for the check).

**How I coded it:** integer points, added up. A page can score 0–11.

| Signal | Condition | Points | Reason code |
|---|---|---|---|
| Staleness | ≥365 days since update | +3 | `stale_365_plus` |
| Staleness | 180–364 days | +2 | `stale_180_plus` |
| Staleness | 90–179 days | +1 | `stale_90_plus` |
| Visibility | ≥5,000 impressions (90d) | +3 | `visible_high` |
| Visibility | 500–4,999 impressions | +2 | `visible_mid` |
| Visibility | 100–499 impressions | +1 | `visible_low` |
| Position risk | position 1–10 and age ≥180d | +2 | `page_one_decay_risk` |
| Position room | position 11–20 | +1 | `striking_distance` |
| Weak capture | CTR < 0.5% and ≥500 impressions | +2 | `low_ctr_visible` |
| Thin but seen | <1,200 words and ≥250 impressions | +1 | `thin_visible` |

**Suggested actions** follow the reason codes, like the reference baseline: `thin_visible`
→ `expand_and_refresh`; `low_ctr_visible` → `refresh_and_review_ctr`; any staleness code →
`refresh`; otherwise `monitor`.

One deliberate difference from the reference rule in `scripts/02_baseline_score.py`: the
reference attaches a `declining_with_demand` reason code using `trend_direction` — the very
column the label is derived from. The reference *score* does not use it, but I left
`trend_direction` out of my rule entirely, codes included, so there is nothing to argue
about: my queue is built only from things a reviewer could know without seeing the outcome.


In [1]:
# --- Setup: find the repo root, load the prepared feature vector -------------------
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd

_root = Path.cwd()
while not (_root / "data" / "raw" / "content_refresh_anonymized.csv").exists() and _root != _root.parent:
    _root = _root.parent
ROOT = _root
sys.path.insert(0, str(ROOT / "scripts"))

from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k

SEED = 42  # fixed everywhere in this notebook, same as the reference pipeline

FEATURE_PATH = ROOT / "data" / "processed" / "refresh_feature_vector.csv"
if not FEATURE_PATH.exists():
    import subprocess
    subprocess.run([sys.executable, str(ROOT / "scripts" / "01_prepare_features.py")], check=True)

fv = pd.read_csv(FEATURE_PATH)
y = fv["is_declining_label"].astype(int)
print(f"Feature vector: {len(fv):,} rows x {len(fv.columns)} cols")
print(f"Declining-label base rate (full data): {y.mean():.3f}")

# The rule, implemented once and reused in every section below.

def my_action_score(df):
    """My hand rule (ML-07): integer points, no fitted weights, no label-derived inputs.
    Returns (score, reason_codes) aligned to df's index."""
    score = pd.Series(0.0, index=df.index)
    codes = pd.Series([[] for _ in range(len(df))], index=df.index, dtype=object)

    def apply(mask, points, code):
        score.loc[mask] += points
        for idx in df.index[mask]:
            codes.at[idx].append(code)

    d = df["days_since_last_update"]; imp = df["impressions_90d"]; pos = df["avg_position"]
    apply(d >= 365, 3, "stale_365_plus")
    apply((d >= 180) & (d < 365), 2, "stale_180_plus")
    apply((d >= 90) & (d < 180), 1, "stale_90_plus")
    apply(imp >= 5000, 3, "visible_high")
    apply((imp >= 500) & (imp < 5000), 2, "visible_mid")
    apply((imp >= 100) & (imp < 500), 1, "visible_low")
    apply((pos > 0) & (pos <= 10) & (df["content_age_days"] >= 180), 2, "page_one_decay_risk")
    apply((pos > 10) & (pos <= 20), 1, "striking_distance")
    apply((df["ctr"] < 0.5) & (imp >= 500), 2, "low_ctr_visible")
    apply((df["word_count"] > 0) & (df["word_count"] < 1200) & (imp >= 250), 1, "thin_visible")
    return score, codes


def suggested_action(codes_str):
    codes = set(codes_str.split("|"))
    if "thin_visible" in codes:
        return "expand_and_refresh"
    if "low_ctr_visible" in codes:
        return "refresh_and_review_ctr"
    if codes & {"stale_365_plus", "stale_180_plus", "stale_90_plus"}:
        return "refresh"
    return "monitor"

fv["my_score"], fv["my_codes"] = my_action_score(fv)
fv["reason_codes_str"] = ["|".join(c) if c else "general_refresh_review" for c in fv["my_codes"]]

print("Score distribution (full 30,000 pages):")
print(fv["my_score"].describe().round(2))
print("\nPages carrying each reason code:")
all_codes = pd.Series([c for codes in fv["my_codes"] for c in codes])
print(all_codes.value_counts())
print(f"\nPages with no signal at all (score 0): {(fv['my_score'] == 0).sum():,}")

Feature vector: 30,000 rows x 52 cols
Declining-label base rate (full data): 0.542


Score distribution (full 30,000 pages):
count    30000.00
mean         3.48
std          2.31
min          0.00
25%          1.00
50%          4.00
75%          5.00
max          9.00
Name: my_score, dtype: float64

Pages carrying each reason code:
low_ctr_visible        14245
visible_mid            10575
stale_90_plus           9171
striking_distance       7273
page_one_decay_risk     7076
visible_high            6151
visible_low             5280
stale_180_plus           169
thin_visible              82
stale_365_plus             5
Name: count, dtype: int64

Pages with no signal at all (score 0): 3,962


## 2. Build the ranked queue (writes the CSV)

*Code the score, rank everything, write work/outputs/baseline_action_score.csv.*

---

I scored all 30,000 pages in the prepared feature vector (the output of
`scripts/01_prepare_features.py`), ranked by score with impressions as the tie-break,
and wrote the queue to `work/outputs/baseline_action_score.csv` — one row per page with
its rank, score, reason codes and suggested action.

A ranked list is only a claim until it is measured, so I also evaluated the rule the
honest way: **on the same client holdout the reference pipeline uses** (seed 42, ~20% of
clients held out — 6 of 32 clients, 2,325 pages — so no client's pages are in both
sides). The code below prints my rule's Precision@20/50/100 on that holdout next to the
reference rule's, recomputed from its own saved queue, plus the holdout base rate
(0.391 — random ranking would score ~0.39 at any K on this slice).

Measured result: my rule reaches **P@20 = 0.50, P@50 = 0.48, P@100 = 0.44** on held-out
clients, against the reference rule's **0.15 / 0.24 / 0.36**. Both hand rules are far
below the models I train in ML-08 (P@50 = 0.68) — that gap is the honest case for a model.


In [2]:
def client_holdout(frame, target, seed=SEED, test_share=0.2):
    """Same split as scripts/03_train_model.py: hold out ~20% of CLIENTS (seed 42),
    so no client's pages appear in both train and test."""
    clients = frame["client_id"].fillna("unknown").astype(str).drop_duplicates().to_numpy()
    rng = np.random.default_rng(seed)
    shuffled = rng.permutation(clients)
    n_test = max(1, int(round(len(shuffled) * test_share)))
    test_clients = set(shuffled[:n_test])
    test_mask = frame["client_id"].astype(str).isin(test_clients).to_numpy()
    return np.arange(len(frame))[~test_mask], np.arange(len(frame))[test_mask], test_clients

train_idx, test_idx, test_clients = client_holdout(fv, y)
print(f"Holdout: {len(test_clients)} clients, {len(test_idx):,} pages; "
      f"base rate on holdout = {y.iloc[test_idx].mean():.3f}")

# Reference rule scores, mapped from its own saved queue (same as scripts/03 does)
ref = pd.read_csv(ROOT / "data" / "processed" / "baseline_refresh_queue.csv")
ref_lookup = ref.set_index("content_id")["baseline_refresh_score"]
ref_test_scores = fv.iloc[test_idx]["content_id"].map(ref_lookup).fillna(0).to_numpy()
my_test_scores = fv.iloc[test_idx]["my_score"].to_numpy()
y_test = y.iloc[test_idx]

rows = []
for name, scores in [("reference_rule (scripts/02)", ref_test_scores), ("my_rule (points)", my_test_scores)]:
    rows.append({"rule": name,
                 "P@20": round(precision_at_k(y_test, scores, 20), 3),
                 "P@50": round(precision_at_k(y_test, scores, 50), 3),
                 "P@100": round(precision_at_k(y_test, scores, 100), 3)})
print(pd.DataFrame(rows).to_string(index=False))
print(f"(random ranking on this holdout would give ~{y_test.mean():.3f} at every K)")

# --- Build and write the full ranked queue -------------------------------------------
queue = fv.copy()
queue["suggested_action"] = queue["reason_codes_str"].apply(suggested_action)
queue = queue.sort_values(["my_score", "impressions_90d"], ascending=False).reset_index(drop=True)
queue.insert(0, "rank", np.arange(1, len(queue) + 1))
out_cols = ["rank", "content_id", "client_id", "my_score", "reason_codes_str", "suggested_action",
            "impressions_90d", "clicks_90d", "sessions_90d", "avg_position", "ctr",
            "content_age_days", "days_since_last_update", "word_count"]
out_path = ROOT / "work" / "outputs" / "baseline_action_score.csv"
out_path.parent.mkdir(parents=True, exist_ok=True)
queue[out_cols].to_csv(out_path, index=False)
print(f"\nWrote {out_path.relative_to(ROOT)}: {len(queue):,} rows")
print("Action mix:", queue["suggested_action"].value_counts().to_dict())

Holdout: 6 clients, 2,325 pages; base rate on holdout = 0.391


                       rule  P@20  P@50  P@100
reference_rule (scripts/02)  0.15  0.24   0.36
           my_rule (points)  0.50  0.48   0.44
(random ranking on this holdout would give ~0.391 at every K)



Wrote work/outputs/baseline_action_score.csv: 30,000 rows
Action mix: {'refresh_and_review_ctr': 14216, 'monitor': 12029, 'refresh': 3673, 'expand_and_refresh': 82}


## 3. Top-20 review

*For each of the top 20: action, reason code, confidence note, and what would make it wrong.*

---

I read the top 20 of my own queue by hand (the table is printed below). Format per pick:
**action — reason codes — my confidence — what would make this pick wrong.** "Declining"
below means the observed label on that page; the rule never saw it.

1. **expand_and_refresh** — all five codes (score 9) — *high confidence* — 5,129 impressions,
   position 6.3, CTR 0.19%, 1,184 words, observed declining. Every signal agrees. Wrong if:
   the 104-day update stamp is the dataset's coarse bucket rather than real neglect.
2. **refresh_and_review_ctr** — stale + visible + page-one risk + low CTR — *high* —
   517K impressions, position 4.2, CTR 0.14%, observed declining. Wrong if: this page's
   results layout structurally suppresses CTR. Note its word_count is 0 = missing, so depth
   cannot be judged either way.
3. **refresh_and_review_ctr** — same four codes — *low confidence, my weakest pick* —
   208K impressions but **0 clicks, 6 sessions, engagement and scroll both 0**. The label
   says declining, but the engagement data looks unmeasured rather than bad. Wrong if:
   this is a tracking gap, not a content problem — check measurement before editing.
4. **refresh_and_review_ctr** — same four codes — *medium* — 201K impressions, position 5.8,
   observed **not** declining. Wrong because: it belongs to the same very large client as
   picks 3, 5, 7–10…; at this client's scale, everything maxes the visibility points.
5. **refresh_and_review_ctr** — same four codes — *medium* — 201K impressions, position 5.7,
   CTR 0.11%, not declining. Picks 4–5 as a pair show the rule firing on client scale,
   not on page decay.
6. **refresh_and_review_ctr** — same four codes — *high* — 187K impressions, position 4.0,
   845 clicks, observed declining.
7. **refresh_and_review_ctr** — same four codes — *medium-low* — 176K impressions,
   position 4.3, not declining; same-client pattern as pick 4.
8. **refresh_and_review_ctr** — same four codes — *medium* — 159K impressions, CTR 0.06%
   (lowest in the top 20), engagement 0. Even though the label disagrees, a CTR review is
   defensible. Wrong if: the 0 engagement is unmeasured, like pick 3.
9. **refresh_and_review_ctr** — same four codes — *low-medium* — 152K impressions,
   position 3.3, engagement 6.18% — comparatively healthy engagement; the rule over-fires
   here. Wrong because: visibility + staleness alone carried it.
10. **refresh_and_review_ctr** — same four codes — *high* — 152K impressions, position 6.5,
    observed declining.
11. **refresh_and_review_ctr** — same four codes — *medium* — 151K impressions, CTR 0.23%,
    word_count missing (0), not declining.
12. **refresh_and_review_ctr** — same four codes — *medium* — 148K impressions, position 3.7,
    CTR 0.48% — just under my 0.5% cutoff; a borderline code shows how sensitive the rule
    is to hand-set thresholds.
13. **refresh_and_review_ctr** — same four codes — *high* — 145K impressions, position 3.3,
    CTR 0.46%, observed declining.
14. **refresh_and_review_ctr** — same four codes — *high* — 142K impressions, position 3.9,
    observed declining.
15. **refresh_and_review_ctr** — same four codes — *medium* — 140K impressions,
    position 5.3, not declining.
16. **refresh_and_review_ctr** — same four codes — *medium-low* — 132K impressions,
    engagement 0.78%, not declining.
17. **refresh_and_review_ctr** — same four codes — *medium* — 128K impressions and a real
    1,764-word page: the depth signal cannot fire, so it ranks on visibility + staleness +
    CTR only. Not declining.
18. **refresh_and_review_ctr** — same four codes — *medium-low* — 127K impressions,
    engagement 0.26%, not declining.
19. **refresh_and_review_ctr** — same four codes — *high* — 126K impressions, position 7.6,
    CTR 0.12%, observed declining.
20. **refresh_and_review_ctr** — same four codes — *medium-low* — 126K impressions,
    1,522 words, engagement 5.25% — a healthy-looking page the rule over-fires on
    visibility alone. Not declining.

**The honest summary of my own top 20:** 8 of 20 are observed declining (0.40) — *below*
the full-data base rate of 0.542, because 15 of the 20 come from one very large client
whose pages max out the visibility points whatever their direction. On held-out clients
the same rule measured P@20 = 0.50 against a 0.391 base rate (Section 2). Both numbers
are real; they measure different slices. The lesson I take into ML-08: a hand rule's
precision@K is hostage to the client mix at the top of the list, and any claim about a
ranked queue has to name its slice.


In [3]:
queue = pd.read_csv(ROOT / "work" / "outputs" / "baseline_action_score.csv")
labels = fv.set_index("content_id")["is_declining_label"]
top20 = queue.head(20).copy()
top20["observed_declining"] = top20["content_id"].map(labels)
show = ["rank", "content_id", "client_id", "my_score", "suggested_action", "observed_declining",
        "impressions_90d", "avg_position", "ctr", "days_since_last_update", "word_count"]
print(top20[show].to_string(index=False))
print(f"\nObserved declining in top 20: {top20['observed_declining'].sum()}/20 "
      f"(full-data base rate: {labels.mean():.3f})")
print("Top-20 client mix:", top20["client_id"].value_counts().to_dict())

 rank           content_id         client_id  my_score       suggested_action  observed_declining  impressions_90d  avg_position  ctr  days_since_last_update  word_count
    1 content_18368918809f client_3fdba35f04       9.0     expand_and_refresh                   1             5129           6.3 0.19                     104      1184.0
    2 content_5fe46e04994d client_4e07408562       8.0 refresh_and_review_ctr                   1           517715           4.2 0.14                     104         0.0
    3 content_c8e9d6ab9013 client_19581e27de       8.0 refresh_and_review_ctr                   1           208678           9.7 0.00                     104         0.0
    4 content_d17681677e69 client_19581e27de       8.0 refresh_and_review_ctr                   0           201584           5.8 0.24                     104         0.0
    5 content_a7427266c305 client_19581e27de       8.0 refresh_and_review_ctr                   0           201111           5.7 0.11                 

## 4. Weak picks + leakage check

*Which picks look wrong and why? Confirm no product flags or future windows leaked in.*

---

**Which picks look wrong, and why.** Three stand out from the review above:

- **Pick 3** (208K impressions, 0 clicks, 6 sessions, engagement/scroll exactly 0): the
  numbers describe a page whose engagement was probably never measured, not a page
  readers hate. My rule cannot tell "zero" from "not measured" — the data dictionary
  warns about exactly this for rate columns.
- **Picks 4–5** (and most of the list): one client supplies 15 of my top 20. The rule
  has no sense of proportion across clients, so "most visible" quietly becomes "the
  biggest client's pages". A production version should probably cap or normalise
  per client — noted as a limitation, not silently fixed after seeing the labels.
- **Picks 9 and 20**: healthy engagement (6.18%, 5.25%) and real word counts, carried
  into the top 20 by visibility + the coarse staleness stamp alone.

A data caveat the review exposed: `days_since_last_update` is extremely coarse in this
slice — the median page sits at 20 days and the 75th percentile at 104, so my three
staleness tiers mostly collapse into two buckets and the +3 tier (≥365 days) fires on
under 1% of pages. The rule still ranks, but "staleness" here is closer to a binary
than a scale. I kept the tiers (they are the honest reading of the field) and disclose
the coarseness instead of pretending precision.

**Leakage check (confirmed in the code below).** The scoring inputs are only:
impressions, average position, CTR, word count, content age, days since last update —
all observable at review time. Specifically:

- `trend_direction` and `trend_pct` are **not** used anywhere in my rule — not in the
  score, not in the reason codes, not in the action mapping. (The label is derived from
  `trend_direction`; using either column would be reading the answer.)
- No product decision flags are used — and none exist in this starter data
  (`health_score`, `needs_ctr_fix`, … were intentionally not shipped, per `DATA_USE.md`).
- No future windows: every input is a trailing-90-day aggregate or a page attribute
  known at the snapshot date. The one shared-window caveat — the label's trend is also
  computed inside that trailing window — is a property of the starter snapshot itself;
  it applies equally to the reference pipeline, and it is why my claims stay
  directional and decision-support rather than predictive.
- `content_id` / `client_id` appear in the CSV for joining and review only, never
  as scoring inputs.


In [4]:
# --- Leakage / sanity audit of my own rule --------------------------------------------
scoring_inputs = ["impressions_90d", "avg_position", "ctr", "word_count",
                  "content_age_days", "days_since_last_update"]
banned = {"trend_direction", "trend_pct", "is_declining_label", "content_id", "client_id"}
assert not (set(scoring_inputs) & banned), "label-derived or ID column in scoring inputs!"
print("Scoring inputs:", scoring_inputs)
print("Banned columns used by the rule: none (asserted)")

product_flags = [c for c in fv.columns if c in
                 {"health_score", "needs_ctr_fix", "is_quick_win", "is_zombie"}]
print("Product decision flags present in starter data:", product_flags or "none (not shipped)")

print("\ndays_since_last_update is coarse — quantiles:")
print(fv["days_since_last_update"].quantile([0.25, 0.5, 0.75, 0.9, 0.99]).round(0))

top20 = pd.read_csv(ROOT / "work" / "outputs" / "baseline_action_score.csv").head(20)
print("\nTop-20 client concentration:", top20["client_id"].value_counts().head(3).to_dict())
print("Rows in top 20 with word_count == 0 (missing, filled):",
      int((top20["word_count"] == 0).sum()))

Scoring inputs: ['impressions_90d', 'avg_position', 'ctr', 'word_count', 'content_age_days', 'days_since_last_update']
Banned columns used by the rule: none (asserted)
Product decision flags present in starter data: none (not shipped)

days_since_last_update is coarse — quantiles:
0.25     20.0
0.50     20.0
0.75    104.0
0.90    104.0
0.99    106.0
Name: days_since_last_update, dtype: float64

Top-20 client concentration: {'client_19581e27de': 15, 'client_4e07408562': 4, 'client_3fdba35f04': 1}
Rows in top 20 with word_count == 0 (missing, filled): 17


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

*Final box left unticked on purpose: the notebook is complete and executed in `work/notebooks/`; committing it to my repo is my own submission step.*